In [ ]:
# ==========================================
# BƯỚC 1: CÀI ĐẶT MÔI TRƯỜNG & THƯ VIỆN
# ==========================================
!pip install pyspark matplotlib seaborn scikit-learn -q

import os
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, mean
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# ==========================================
# BƯỚC 2: KHỞI TẠO SPARK & ĐỌC DỮ LIỆU
# ==========================================
print("⏳ Khởi tạo Spark Session...")
spark = SparkSession.builder.appName("Depression_KFold_Kaggle").getOrCreate()

# ĐƯỜNG DẪN FILE DATASET TRÊN KAGGLE
CSV_FILE = "/kaggle/input/datasets/akina484/atn-data-1/Student Depression Dataset.csv"

print(f" Đang đọc file từ: {CSV_FILE}")

try:
    if not os.path.exists(CSV_FILE):
        print(" Lỗi: Không tìm thấy file. Hãy kiểm tra lại đường dẫn dataset trên Kaggle.")
    else:
        df = spark.read.csv(CSV_FILE, sep=';', header=True, inferSchema=True)
        
        # Xóa khoảng trắng thừa ở tên cột nếu có
        for col_name in df.columns:
            df = df.withColumnRenamed(col_name, col_name.strip())
            
        print(" Đọc dữ liệu thành công!")

        # ==========================================
        # BƯỚC 3: XỬ LÝ DỮ LIỆU (PREPROCESSING)
        # ==========================================
        # 1. Chuyển đổi dữ liệu chữ sang số
        if "Gender" in df.columns:
            df = df.withColumn("Gender", when(col("Gender") == "Male", 1).otherwise(0))

        if "Family History of Mental Illness" in df.columns:
            df = df.withColumn("Family History of Mental Illness", 
                               when(col("Family History of Mental Illness") == "Yes", 1).otherwise(0))

        if "Sleep Duration" in df.columns:
            df = df.withColumn("Sleep Duration",
                when(col("Sleep Duration") == "Less than 5 hours", 0)
                .when(col("Sleep Duration") == "5-6 hours", 1)
                .when(col("Sleep Duration") == "7-8 hours", 2)
                .otherwise(3)
            )

        # 2. Xử lý Missing Value
        input_cols = ["Age", "CGPA", "Academic Pressure", "Study Satisfaction", 
                      "Financial Stress", "Sleep Duration", "Gender", "Family History of Mental Illness"]
        existing_cols = [c for c in input_cols if c in df.columns]

        # Đảm bảo các cột là số thực (đề phòng lỗi parse từ csv)
        for column in existing_cols:
            df = df.withColumn(column, col(column).cast("double"))
            mean_val = df.select(mean(col(column))).collect()[0][0]
            if mean_val is not None:
                df = df.fillna(mean_val, subset=[column])

        # Đảm bảo cột Target là số nguyên
        if "Depression" in df.columns:
            df = df.withColumn("Depression", col("Depression").cast("integer"))
        else:
            raise ValueError("Không tìm thấy cột 'Depression'.")

        # ==========================================
        # BƯỚC 4: THIẾT LẬP K-FOLD & HUẤN LUYỆN
        # =========================================
        print("\n Bắt đầu thiết lập Pipeline và K-Fold (5 Folds)...")
        
        assembler = VectorAssembler(inputCols=existing_cols, outputCol="features_raw")
        scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)
        lr = LogisticRegression(featuresCol="features", labelCol="Depression", maxIter=100)
        
        pipeline = Pipeline(stages=[assembler, scaler, lr])

        # Random chia dữ liệu thành 5 phần (folds)
        folds = df.randomSplit([0.2, 0.2, 0.2, 0.2, 0.2], seed=42)
        
        results_metrics = []
        plt.figure(figsize=(10, 8))
        
        evaluator_auc = BinaryClassificationEvaluator(labelCol="Depression", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
        evaluator_multi = MulticlassClassificationEvaluator(labelCol="Depression", predictionCol="prediction")

        for i in range(5):
            print(f"Đang xử lý Fold {i+1}...")
            # Lấy fold i làm test, các fold còn lại gộp làm train
            test_data = folds[i]
            train_data = spark.createDataFrame(spark.sparkContext.emptyRDD(), schema=df.schema)
            for j in range(5):
                if j != i:
                    train_data = train_data.union(folds[j])
            
            # Huấn luyện
            model = pipeline.fit(train_data)
            
            # Dự đoán
            predictions = model.transform(test_data)
            
            # Tính toán các chỉ số
            auc_val = evaluator_auc.evaluate(predictions)
            accuracy = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "accuracy"})
            f1 = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "f1"})
            
            # Tính Precision & Recall (cho lớp 1)
            tp = predictions.filter("prediction = 1 AND Depression = 1").count()
            fp = predictions.filter("prediction = 1 AND Depression = 0").count()
            fn = predictions.filter("prediction = 0 AND Depression = 1").count()
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            
            results_metrics.append({
                'Fold': f"Fold {i+1}",
                'Samples': test_data.count(),
                'Accuracy': accuracy,
                'Precision': precision,
                'Recall': recall,
                'F1-Score': f1,
                'AUC': auc_val
            })
            
            # Lấy xác suất dự đoán để vẽ ROC
            probs_and_labels = predictions.select("probability", "Depression").collect()
            y_prob = [row['probability'][1] for row in probs_and_labels]
            y_test = [row['Depression'] for row in probs_and_labels]
            
            fpr, tpr, _ = roc_curve(y_test, y_prob)
            plt.plot(fpr, tpr, lw=2, label=f"Fold {i+1} (AUC = {auc_val:.4f})")

        # ==========================================
        # BƯỚC 5: IN BẢNG KẾT QUẢ
        # ==========================================
        res_df = pd.DataFrame(results_metrics)
        
        avg_row = {
            'Fold': 'AVG', 'Samples': '---',
            'Accuracy': res_df['Accuracy'].mean(),
            'Precision': res_df['Precision'].mean(),
            'Recall': res_df['Recall'].mean(),
            'F1-Score': res_df['F1-Score'].mean(),
            'AUC': res_df['AUC'].mean()
        }
        std_row = {
            'Fold': 'STD', 'Samples': '---',
            'Accuracy': res_df['Accuracy'].std(),
            'Precision': res_df['Precision'].std(),
            'Recall': res_df['Recall'].std(),
            'F1-Score': res_df['F1-Score'].std(),
            'AUC': res_df['AUC'].std()
        }
        
        res_df = pd.concat([res_df, pd.DataFrame([avg_row, std_row])], ignore_index=True)

        print("\n+---------------------------------------------------------------------------------+")
        print("| Fold     | Samples   | Accuracy  | Precision | Recall    | F1-Score  | AUC      |")
        print("+---------------------------------------------------------------------------------+")
        for index, row in res_df.iterrows():
            if row['Fold'] in ['AVG', 'STD']:
                print("+---------------------------------------------------------------------------------+")
            print(f"| {row['Fold']:<8} | {row['Samples']:<9} | {row['Accuracy']:.4f}    | {row['Precision']:.4f}    | {row['Recall']:.4f}    | {row['F1-Score']:.4f}    | {row['AUC']:.4f}   |")
        print("+---------------------------------------------------------------------------------+")

        # ==========================================
        # BƯỚC 6: HOÀN THIỆN BIỂU ĐỒ & LƯU MODEL
        # ==========================================
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate (FPR)', fontsize=12)
        plt.ylabel('True Positive Rate (TPR)', fontsize=12)
        plt.title('ROC Curves for 5-Folds (PySpark Logistic Regression)', fontsize=14, fontweight='bold')
        plt.legend(loc="lower right", fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.show()
        
        # Train lại model cuối cùng trên toàn bộ dữ liệu để lưu
        print("\n⏳ Đang huấn luyện lại mô hình trên toàn bộ dữ liệu để xuất file...")
        final_model = pipeline.fit(df)
        
        # Lưu model xuống /kaggle/working/ để tải về
        save_path = '/kaggle/working/spark_depression_model'
        final_model.write().overwrite().save(save_path)
        print(f" Đã lưu model tại: {save_path}")
        print(" Bạn hãy check cột Output bên phải Kaggle để tải thư mục model về nhé!")

except Exception as e:
    print(f" Lỗi xử lý: {e}")
finally:
    spark.stop()